# a2_seq

> Phase 21 (`specs/multiagent/phase21/`): A2's experience screen, board by board. `TargetedLiar` lies, on `ClueLiar`'s cadence, on clues the success guards hold and never on clues the failure guards hold (P21-2); `recordLines` writes each honest slot's board line and, from board 3, its arm's record (P21-4); `boardCell21` builds board b of a team's sequence (boards 1-2: assigned guards and targeted liars, the same cell in every arm; boards 3-6: the volunteer rule, `ClueLiar`s and the arm's record); `runBoard21` plays a board as `colony_trace.replay` does, with the targeted liars and a `ContestLog` that pays no one (its watchers), and returns Stage 3's, Phase 19's and A2's columns; `runSequences21` plays the teams' sequences in six waves.

In [ ]:
#| default_exp a2_seq

In [ ]:
#| hide
from nbdev.showdoc import *

## Targeted liars

On a lying turn (every other turn, as `ClueLiar`) a targeted liar picks, among the clues of the open subtasks other than its own, those a `hit` slot holds and no `avoid` slot holds; if there is none, those no `avoid` slot holds. It never lies on a clue an `avoid` slot holds. The false residue is drawn as `ClueLiar` draws it.

In [ ]:
#| export
import json
from pathlib import Path

from explRL.a2_cols import LIARS, a2Columns, failureIds, seating, successIds
from explRL.clue_agents import ClueLiar, RuleAgent
from explRL.clue_fix import fixOptions
from explRL.clue_runs import clueMetrics, clueSetup, stage4Members
from explRL.clue_text import TextClueEnv
from explRL.colony3 import ARMS3, stage3Columns
from explRL.colony_agents import faultMembers
from explRL.colony_grid import colonyCell
from explRL.colony_trace import Replay
from explRL.contest import ContestLog
from explRL.contest_eval import contestColumns
from explRL.llm_agent import LLMMember, runTurns
from explRL.llm_budget import estimateCost
from explRL.llm_runs import llmStats, runLLMCell
from explRL.multi import spawnRngs


class TargetedLiar(ClueLiar):
    "ClueLiar's cadence; lies on a clue a `hit` slot holds and no `avoid` slot holds, else on any clue no `avoid` slot holds."
    def __init__(self, channel, rng, hit=(), avoid=()):
        super().__init__(channel, rng)
        self.hit, self.avoid = frozenset(hit), frozenset(avoid)

    def candidates(self, b):
        "The (k, j) it may lie on now: the preferred ones (a hit slot holds), else every clue no avoid slot holds."
        ok = [(k, j) for k in b.open if k != b.opened[self.id] for j, hs in enumerate(b.tasks[k].holders) if not self.avoid & set(hs)]
        return [(k, j) for k, j in ok if self.hit & set(b.tasks[k].holders[j])] or ok

    def choose(self, b, t):
        self.turn += 1
        cand = self.candidates(b) if self.channel is not None and self.turn % 2 else []
        if cand:
            k, j = cand[int(self.rng.integers(len(cand)))]
            p, r = b.tasks[k].task.clue(j)
            return ('share', (k, j, int((r + 1 + self.rng.integers(p - 1)) % p)))
        return RuleAgent.choose(self, b, t)

## Records (P21-4)

Every honest slot gets the board line. From board 3 the `shared` arm adds the team's totals per earlier board and the `own` arm the agent's own contests and the correct answers that followed them; `none` adds nothing. `history` holds the team's finished boards 1 to b-1 (`historyEntry` of each board's row), so a record follows the identity whatever slot it sits in. The texts are frozen in `A2_PLAN` (`TEXTS21`).

In [ ]:
#| export
TEXTS21 = dict(board='This is board {b} of 6 played by the same team.',
               shared='Team record on the earlier boards: {items}.', own='Your record on the earlier boards: {items}.',
               item='board {n}: {who} contested {c}, and teammates then answered those subtasks correctly {a}',
               who=dict(shared='teammates', own='you'))


def _count(n, word):
    return f"{n} {word}{'s' * (n != 1)}"


def historyEntry(row):
    "A finished board's record entry from its row: {board, contests, followed}, per identity."
    return dict(board=int(row['pos']) + 1, contests=json.loads(row['idContests']), followed=json.loads(row['idFollowed']))


def recordLines(arm, history, b, team):
    "((slot, text), ...) for the honest slots of board b of `team` in `arm` (none, shared, own); history: boards 1..b-1."
    T, seat, out = TEXTS21, seating(team, b), []
    if b >= 3 and arm != 'none':
        assert [h['board'] for h in history] == list(range(1, b)), 'history must hold boards 1..b-1'
    for s, a in enumerate(seat):
        txt = T['board'].format(b=b)
        if b >= 3 and arm != 'none':
            nums = [(h['board'], h['contests'][a], h['followed'][a]) if arm == 'own' else
                    (h['board'], sum(h['contests']), sum(h['followed'])) for h in history]
            items = '; '.join(T['item'].format(n=n, who=T['who'][arm], c=_count(c, 'false value'), a=_count(x, 'time'))
                              for n, c, x in nums)
            txt += ' ' + T[arm].format(items=items)
        out.append((LIARS + s, txt))
    return tuple(out)


def boardCell21(team, b, arm, history=()):
    """Board b of `team`'s sequence (seed 10 * team + b). Boards 1-2: Stage 1's colonyGuards cell (4 guards), targeted
    liars against the seated success and failure guards, the board line; level 'common|b<b>', the same in every arm.
    Boards 3-6: Stage 3's colonyVolunteer cell (ClueLiars), the arm's record; level '<arm>|b<b>'."""
    seed, seat = 10 * team + b, seating(team, b)
    slot = {a: LIARS + s for s, a in enumerate(seat)}
    if b <= 2:
        c = colonyCell('a2', 'colonyGuards', 16, seed, 'independent')
        extra = dict(targets=[sorted(slot[a] for a in successIds(team)), sorted(slot[a] for a in failureIds(team))])   # [hit, avoid]
    else:
        c, extra = colonyCell('a2', 'colonyVolunteer', 16, seed, 'independent', arms=ARMS3), {}
    c['cfg']['world'].update(team=team, board=b, seat=seat, slotLines=recordLines('none' if b <= 2 else arm, history, b, team), **extra)
    c['level'] = f"{'common' if b <= 2 else arm}|b{b}"
    return c

## The board runner

`replay21` is `colony_trace.replay` for an LLM cell, with the targeted liars when the world has `targets` and a `ContestLog` attached to the board for its watchers (its contest slots are empty, so it pays no one and every reward is unchanged). Without `targets` its rows and turn log are `runClue`'s.

In [ ]:
#| export
def replay21(cfg, seed, client):
    "colony_trace.replay (an LLM cell) with TargetedLiars from world['targets'] ([hit, avoid] slots) and a ContestLog that pays no one -> Replay."
    T, nL, w = cfg.turnsPerAgent(), cfg.nLiar, cfg.world
    b, ch = clueSetup(cfg, seed)
    b.contestLog = ContestLog(b, range(nL), ())                   # watchers only: no contest slot, nothing paid
    client.reserve(estimateCost(cfg.model, 'clue', cfg.N - nL * (w.get('fault') != 'adaptive'), T))
    env = TextClueEnv(b, w.get('hint'), w.get('showOpen', False), **fixOptions(w), nLiar=nL)
    mk = lambda: LLMMember(client, env, ch, cfg.memory, cfg.framing)
    rngs, tg = spawnRngs(seed + 3000, nL), w.get('targets')
    liars = [TargetedLiar(ch, r, *tg) for r in rngs] if tg else faultMembers(w, nL, ch, rngs, mk)
    members = liars + [mk() for _ in range(cfg.N - nL)]
    llm = [m for m in members if isinstance(m, LLMMember)]
    hist = runTurns(stage4Members(w, members), b, T)
    row = dict(episode=0, **clueMetrics(cfg, b, hist, range(nL, cfg.N), T), **llmStats(llm))
    log = [dict(episode=0, **x) for m in llm for x in m.log]
    subs = {(s[0], s[1]) for s in b.submits}
    ans = {(float(r.t), r.agent): r.extras[0] for r in hist.records if (float(r.t), r.agent) in subs and not isinstance(members[r.agent], LLMMember)}
    ans.update({(float(x['t']), x['agent']): x['action'][1] for x in log if x['action'][0] == 'submit'})
    return Replay([row], log, b, ans, nL, T)


def runBoard21(cfg, seed, client):
    "runClue's signature: the board's row with Stage 3's, Phase 19's (contest_eval.contestColumns) and A2's columns, and its turn log."
    rp = replay21(cfg, seed, client)
    return [{**stage3Columns(cfg, rp, seed), **contestColumns(cfg, rp.board, [rp.log]), **a2Columns(cfg, rp.board)}], rp.log


def runBoard21Cell(cell):
    "For runWorkload (picklable): one A2 board through runBoard21."
    return runLLMCell(cell, run=runBoard21)

## Sequences

Six waves by board position. Waves 1-2 run one cell per team (keyed without the arm) and copy its row into each arm, since three parallel live calls of the same request would all be paid (the response cache stores a reply only after its call). Wave b >= 3 runs board b of every team and arm, its record built from that arm's boards 1 to b-1. Rows go to `runs.csv` (with `arm`), each board's lines to `records.jsonl` and the turn logs to `turns/`; a rerun skips every board already in `runs.csv`, so an interrupted run resumes. `alongside` grids (P2, P21-14) join the first wave that runs, in the same pool, and their rows go to `<name>.csv` once every cell has come back (a grid whose file exists is not run again; a partial one is, its finished boards from the cache); `gate(rows)` after wave 2 returns the reasons to stop before wave 3 (none: go on).

In [ ]:
#| export
def _key(r):
    return int(float(r['seqId'])), str(r['arm']), int(float(r['pos'])) + 1


def runSequences21(teams, arms, spec, out, workers=8, boards=6, edit=None, alongside=(), gate=None, run=None):
    """Play the teams' sequences in waves (boards 1..`boards`) through runWorkload -> dict(rows, alongside, ran, stopped);
    edit(cell) may change cells (tests: fewer turns); rows of earlier runs in out/runs.csv are kept and not replayed."""
    from explRL.bench_grid import readCSV, writeCSV
    from explRL.llm_sched import runWorkload
    from explRL.a2_seq import runBoard21Cell             # the module's (picklable for spawned workers, also from a notebook)
    out, run = Path(out), run or runBoard21Cell
    rows = readCSV(out / 'runs.csv') if (out / 'runs.csv').exists() else []
    done, ran, side, stopped = {_key(r) for r in rows}, [], {}, None
    for b in range(1, boards + 1):
        hist = lambda t, a: [historyEntry(r) for r in sorted((r for r in rows if _key(r)[:2] == (t, a) and _key(r)[2] < b), key=_key)]
        want = [(t, None if b <= 2 else a) for t in teams for a in ([None] if b <= 2 else arms)
                if (b <= 2 and any((t, x, b) not in done for x in arms)) or (b > 2 and (t, a, b) not in done)]
        cells = [edit(c) if edit else c for c in (boardCell21(t, b, a, hist(t, a) if a else ()) for t, a in want)]
        grids = [('a2', cells)] + [(n, g) for n, g in alongside if not (out / f'{n}.csv').exists()]
        if cells:
            res = runWorkload(grids, spec, workers, str(out / 'turns'), run=run)
            ran.append((b, len(cells)))
            studies = {n: {c['study'] for c in g} for n, g in grids}
            side.update({n: [r for r in res['rows'] if r['study'] in studies[n]] for n, _ in grids[1:]})
            for n, g in grids[1:]:                       # only a complete grid is saved; a partial one runs again (cached: $0)
                if len(side[n]) == len(g):
                    writeCSV(side[n], out / f'{n}.csv')
                else:
                    stopped = stopped or f'{n}: {len(g) - len(side[n])} boards missing'
            new = [r for r in res['rows'] if r['study'] == 'a2']
            if res['stopped'] or len(new) < len(cells):
                stopped = res['stopped'] or f'wave {b}: {len(cells) - len(new)} boards missing'
            rows += [dict(r, arm=a) for r in new for a in (arms if b <= 2 else [r['level'].split('|')[0]])]
            with open(out / 'records.jsonl', 'a') as f:
                for c in cells:
                    w = c['cfg']['world']
                    f.write(json.dumps(dict(team=w['team'], board=b, level=c['level'], lines=w['slotLines'])) + '\n')
            done = {_key(r) for r in rows}
            writeCSV(rows, out / 'runs.csv')
        if stopped or (b == 2 and gate is not None and (stopped := '; '.join(gate(rows)) or None)):
            break
    return dict(rows=rows, alongside=side, ran=ran, stopped=stopped)

## Tests

In [ ]:
# test_targeted (AC-4, FR-A3): on a full 16-agent board 1 with rule agents (numpy), no lie lands on a clue a failure guard
# holds; whenever a clue a success guard holds (and no failure guard) is open, the lie lands on one; the liars lie on
# exactly ClueLiar's turns (every odd turn with a candidate)
import json, tempfile
from explRL.a2_cols import guardIds
from explRL.colony_agents import memberClass
from explRL.llm_runs import LLMConfig, makeClient
from explRL.clue_runs import runClue

class Watched(TargetedLiar):
    def choose(self, b, t):
        cand = self.candidates(b) if self.turn % 2 == 0 else None          # the next turn is odd: it will lie if it can
        a = super().choose(b, t)
        self.seen.append((self.turn, cand, a))
        return a

for team, bb in ((2190, 1), (2100, 2), (2103, 1)):
    c = boardCell21(team, bb, 'own')
    cfg, w = LLMConfig.fromDict(c['cfg']), c['cfg']['world']
    assert set(w['targets'][0]) | set(w['targets'][1]) == set(range(4, 8))
    board, ch = clueSetup(cfg, c['seed'])
    rngs = spawnRngs(c['seed'] + 3000, 4)
    liars = [Watched(ch, r, *w['targets']) for r in rngs]
    for m in liars:
        m.seen = []
    members = liars + [memberClass(w, i, 4)(ch, r) for i, r in zip(range(4, 16), spawnRngs(c['seed'] + 1000, 16)[4:])]
    runTurns(members, board, 40)
    hit, avoid, n = set(w['targets'][0]), set(w['targets'][1]), 0
    for m in liars:
        for turn, cand, a in m.seen:
            lied = a[0] == 'share'
            assert lied == bool(turn % 2 and cand), (turn, cand, a)        # ClueLiar's cadence
            if lied:
                k, j, _ = a[1]
                hs = set(board.tasks[k].holders[j])
                assert not hs & avoid and (k, j) in cand
                n += 1
                if any(hit & set(board.tasks[kk].holders[jj]) for kk, jj in cand):
                    assert hs & hit
    assert n > 20
    assert not any(set(board.tasks[s[2]].holders[s[3]]) & avoid for s in board.shareLog if s[1] < 4)

In [ ]:
# test_records21 (AC-6): the board line on boards 1-2 in every arm (no record); from board 3 the own line counts only the
# agent's own contests and followers, the shared line the team totals (the same for every slot, no identity), none only
# the board line; boards 1-2 are identical cells in the three arms; records follow the identity across seatings
hist = [dict(board=1, contests=[0, 2] + [0] * 10, followed=[0, 1] + [0] * 10),
        dict(board=2, contests=[1] + [0] * 10 + [3], followed=[1] + [0] * 10 + [2])]
for b in (1, 2):
    cs = [boardCell21(2100, b, a) for a in ('none', 'shared', 'own')]
    assert cs[0] == cs[1] == cs[2] and cs[0]['level'] == f'common|b{b}'
    assert {t for _, t in cs[0]['cfg']['world']['slotLines']} == {f'This is board {b} of 6 played by the same team.'}
none, shared, own = (dict(recordLines(a, hist, 3, 2100)) for a in ('none', 'shared', 'own'))
assert set(none) == set(range(4, 16)) and set(none.values()) == {'This is board 3 of 6 played by the same team.'}
assert set(shared.values()) == {'This is board 3 of 6 played by the same team. Team record on the earlier boards: board 1: '
    'teammates contested 2 false values, and teammates then answered those subtasks correctly 1 time; board 2: teammates '
    'contested 4 false values, and teammates then answered those subtasks correctly 3 times.'}
seat = seating(2100, 3)
for s, a in enumerate(seat):
    want = {0: (0, 0, 1, 1), 1: (2, 1, 0, 0), 11: (0, 0, 3, 2)}.get(a, (0, 0, 0, 0))
    assert own[4 + s] == ('This is board 3 of 6 played by the same team. Your record on the earlier boards: board 1: you '
        f"contested {_count(want[0], 'false value')}, and teammates then answered those subtasks correctly {_count(want[1], 'time')}; "
        f"board 2: you contested {_count(want[2], 'false value')}, and teammates then answered those subtasks correctly {_count(want[3], 'time')}."), (a, own[4 + s])
c3 = {a: boardCell21(2100, 3, a, hist) for a in ('none', 'shared', 'own')}
assert {c['level'] for c in c3.values()} == {'none|b3', 'shared|b3', 'own|b3'} and 'targets' not in c3['own']['cfg']['world']
strip = lambda c: {k: v for k, v in c['cfg']['world'].items() if k != 'slotLines'}
assert strip(c3['none']) == strip(c3['shared']) == strip(c3['own']) and c3['own']['cfg']['world']['volunteer'] is True
try:
    recordLines('own', hist[:1], 3, 2100)
    raise AssertionError('a gap in the history must fail')
except AssertionError as e:
    assert 'history' in str(e)

In [ ]:
# test_board21 (FR-A4): without targets, runBoard21's row holds runClue's row and its turn log is runClue's (the fake
# client, a short volunteer board); with targets (board 1), the liars never post on a failure guard's clue, and the
# row carries A2's, Stage 3's and Phase 19's columns with the identity keys
from explRL.colony3 import ARMS3
c = colonyCell('t', 'colonyVolunteer', 16, 1800, 'independent', arms=ARMS3)
c['cfg']['turns'] = 3
cfg = LLMConfig.fromDict(c['cfg'])
with tempfile.TemporaryDirectory() as d:
    r0, l0 = runClue(cfg, 1800, makeClient(cfg, dict(kind='fake', cache=d)))
    r1, l1 = runBoard21(cfg, 1800, makeClient(cfg, dict(kind='fake', cache=d)))
    assert l0 == l1 and all(r1[0][k] == v or (v != v and r1[0][k] != r1[0][k]) for k, v in r0[0].items())
    c1 = boardCell21(2190, 1, 'own')
    c1['cfg']['turns'] = 6
    cfg1 = LLMConfig.fromDict(c1['cfg'])
    rows, log = runBoard21(cfg1, c1['seed'], makeClient(cfg1, dict(kind='fake', cache=d)))
r = rows[0]
assert r['liarOnFailure'] == 0 and r['pos'] == 0 and r['seqId'] == 2190 and {'volunteers', 'specIndex', 'misinformedFirst', 'idGuarding'} <= set(r)
assert sorted(json.loads(r['agents'])[4:]) == list(range(4, 16)) and json.loads(r['agents'])[:4] == [0, 1, 2, 3]
assert len({x['agent'] for x in log}) == 12 and all(x['agent'] >= 4 for x in log)

In [ ]:
# test_sequences21 (AC-7, FR-B4, P21-14): with the fake client, 2 teams x 3 arms play waves; boards 1-2 run one cell per
# team and are copied into each arm; board b's record comes from that arm's boards 1..b-1; a resume after board 4 runs
# only boards 5-6; P2's cells run once, in the first wave's pool; a gate after wave 2 stops before wave 3
from pathlib import Path
from explRL.bench_grid import readCSV
from explRL.fix_grid import fixCell

short = lambda c: dict(c, cfg=dict(c['cfg'], turns=2))
arms = ('none', 'shared', 'own')
p2 = [short(dict(fixCell('a2drop', 'baseline', 8, 900, 2), level='drop|liar'))]
with tempfile.TemporaryDirectory() as d:
    spec = dict(kind='fake', cache=str(Path(d) / 'cache'))
    R = runSequences21([2100, 2101], arms, spec, Path(d) / 'seq', workers=2, boards=4, edit=short, alongside=[('p2', p2)])
    assert R['ran'] == [(1, 2), (2, 2), (3, 6), (4, 6)] and R['stopped'] is None and len(R['alongside']['p2']) == 1
    rows = R['rows']
    assert len(rows) == 2 * 3 * 4 and {_key(r) for r in rows} == {(t, a, b) for t in (2100, 2101) for a in arms for b in range(1, 5)}
    for t in (2100, 2101):
        for b in (1, 2):
            same = [r for r in rows if _key(r)[0] == t and _key(r)[2] == b]
            assert len({json.dumps({k: v for k, v in r.items() if k != 'arm'}, sort_keys=True, default=str) for r in same}) == 1
    R2 = runSequences21([2100, 2101], arms, spec, Path(d) / 'seq', workers=2, boards=6, edit=short, alongside=[('p2', p2)])
    assert R2['ran'] == [(5, 6), (6, 6)] and R2['alongside'] == {} and len(R2['rows']) == 2 * 3 * 6
    assert len(readCSV(Path(d) / 'seq' / 'p2.csv')) == 1
    disk = readCSV(Path(d) / 'seq' / 'runs.csv')
    assert {_key(r) for r in disk} == {_key(r) for r in R2['rows']} and len(disk) == 36
    recs = [json.loads(x) for x in (Path(d) / 'seq' / 'records.jsonl').read_text().splitlines()]
    assert len(recs) == 2 + 2 + 6 * 4
    for rec in recs:
        t, b, arm = rec['team'], rec['board'], rec['level'].split('|')[0]
        h = [historyEntry(r) for r in sorted((r for r in disk if _key(r)[:2] == (t, 'own' if arm == 'common' else arm) and _key(r)[2] < b), key=_key)]
        assert [tuple(x) for x in rec['lines']] == list(recordLines('none' if b <= 2 else arm, h, b, t)), rec
    from explRL.llm_sched import runWorkload as _rw
    import explRL.llm_sched as _ls
    def lossy(grids, *a, **k):                    # a run that loses the P2 board (as a cap refusal would)
        res = _rw(grids, *a, **k)
        return dict(res, rows=[r for r in res['rows'] if r['study'] == 'a2'])
    _ls.runWorkload = lossy
    try:
        R4 = runSequences21([2103], arms, spec, Path(d) / 'lossy', workers=2, boards=1, edit=short, alongside=[('p2', p2)])
    finally:
        _ls.runWorkload = _rw
    assert R4['stopped'] == 'p2: 1 boards missing' and not (Path(d) / 'lossy' / 'p2.csv').exists()
    R5 = runSequences21([2103], arms, spec, Path(d) / 'lossy', workers=2, boards=2, edit=short, alongside=[('p2', p2)])
    assert R5['ran'] == [(2, 1)] and len(R5['alongside']['p2']) == 1 and (Path(d) / 'lossy' / 'p2.csv').exists()
    R3 = runSequences21([2102], arms, spec, Path(d) / 'gated', workers=2, edit=short, gate=lambda rows: ['manipulation check failed'])
    assert R3['ran'] == [(1, 1), (2, 1)] and R3['stopped'] == 'manipulation check failed' and len(R3['rows']) == 6

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()